B1. Top customers per nation (8 points)

For each nation, return the top 2 customers by total spend, where total spend is the sum of o_totalprice across all of a customer’s orders.

Columns: n_name, c_custkey, total_spend, rank_in_nation
Ties must be handled deterministically
Order the output by n_name, then rank_in_nation

Paste the first 6 rows of output. Then show the expected number of rows (with a one-line reason) and the actual row count, using COUNT(*).

In [0]:
%sql
WITH customer_spend AS (
    SELECT c.c_custkey, n.n_name, SUM(o.o_totalprice) AS total_spend
    FROM samples.tpch.orders AS o
    JOIN samples.tpch.customer AS c
    ON o.o_custkey = c.c_custkey
    JOIN samples.tpch.nation AS n
    ON c.c_nationkey = n.n_nationkey
    GROUP BY c.c_custkey, n.n_name
),
ranked_customers AS(
    SELECT n_name, c_custkey, total_spend,
    ROW_NUMBER() OVER(PARTITION BY n_name ORDER BY total_spend DESC, c_custkey) AS rank_in_nation
    FROM customer_spend
)
SELECT n_name, c_custkey, total_spend, rank_in_nation
FROM ranked_customers
WHERE rank_in_nation <=2
ORDER BY n_name, rank_in_nation, c_custkey


In [0]:
%sql
SELECT count(DISTINCT n_name) as Total_Countries FROM samples.tpch.nation

B2. Running totals and gaps for one customer (9 points)

(a) 2 points. Write a query that finds customers who placed 2 or more orders on the same date. Pick one of those customers for parts (b) and (c).
(b) 5 points. For that customer, return every order with these columns:
o_orderdate, o_orderkey, o_totalprice
rows_total: a running total, one order at a time, with ties handled
range_total: a running total that treats same-date orders as one step
prev_order_date
days_since_prev
Paste the output. Then, in one sentence, identify the rows where rows_total and range_total differ and explain why.
(c) 2 points. Prove that the final rows_total is correct, by showing expected vs. actual.

In [0]:
%sql
SELECT c.c_custkey, o.o_orderdate, count(o.o_orderkey) AS total_orders
FROM samples.tpch.customer AS c
JOIN samples.tpch.orders AS o
ON c.c_custkey = o.o_custkey
GROUP BY c.c_custkey, o.o_orderdate
ORDER BY total_orders DESC
LIMIT 10;

In [0]:
%sql
SELECT o_orderdate, o_orderkey, o_totalprice, 
SUM(o_totalprice) OVER (ORDER BY o_orderdate, o_orderkey ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS row_total, 
SUM(o_totalprice) OVER(ORDER BY o_orderdate RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW ) AS range_total,
LAG(o_orderdate) OVER(PARTITION BY o_custkey ORDER BY o_orderdate,o_orderkey) AS prev_order_date,
DATEDIFF(o_orderdate,prev_order_date) as days_since_prev
FROM samples.tpch.orders
WHERE o_custkey = 87499
ORDER BY o_orderdate, o_orderkey;

In [0]:
%sql
SELECT SUM(o_totalprice) FROM samples.tpch.orders
WHERE o_custkey = 87499;

B3. Order priority report for Finance (8 points)

For orders placed in 1997, return one row per o_orderpriority with these columns:

order_count
total_revenue: the sum of o_totalprice
line_item_count: the total number of line items in those orders

The revenue must not be inflated. Then reconcile three totals, showing expected vs. actual for each:

The sum of order_count across priorities vs. the number of 1997 orders
The sum of total_revenue vs. the total o_totalprice of 1997 orders
The sum of line_item_count vs. the number of line items belonging to 1997 orders

In [0]:
%sql
SELECT COUNT(o_orderkey) AS order_count, SUM(o_totalprice) as total_revenue, COUNT(l_linenumber)  as line_item_count
FROM samples.tpch.orders
JOIN samples.tpch.lineitem
ON o_orderkey = l_orderkey
WHERE year(o_orderdate) = 1997
GROUP BY o_orderpriority
ORDER BY o_orderpriority ASC;